# DEMO TRÊN REVIEW MỚI

In [1]:
# ============================================================
# TÁC VỤ 1: CÀI THƯ VIỆN + MOUNT DRIVE + IMPORT + KIỂM TRA GPU
# ============================================================

!pip install -q transformers py_vncorenlp sentencepiece pandas torch

from google.colab import drive
drive.mount('/content/drive')

import os
import re
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from transformers import AutoTokenizer, AutoModel
import py_vncorenlp

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("✅ Khởi tạo môi trường hoàn tất")
print("DEVICE:", DEVICE)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ Đang chạy bằng CPU")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.9 MB/s eta 0:00:00
Mounted at /content/drive
✅ Khởi tạo môi trường hoàn tất
DEVICE: cuda
GPU: Tesla T4


In [2]:
# ============================================================
# TÁC VỤ 2: KHAI BÁO PATH CHECKPOINT + VNCORE, KIỂM TRA TỒN TẠI
# ============================================================

PHOBERT_CKPT_PATH = (
    "/content/drive/MyDrive/DACS/"
    "phobert_absa_final_loss_balanced/final_artifacts/"
    "phobert_absa_multiaspect_loss_balanced.pt"
)

VISOBERT_CKPT_PATH = (
    "/content/drive/MyDrive/DACS/"
    "visobert_absa_final_loss_balanced/final_artifacts/"
    "visobert_absa_multiaspect_loss_balanced.pt"
)

VNCORE_DIR = "/content/drive/MyDrive/DACS/My_NLP_Models/vncorenlp"

required_paths = {
    "PhoBERT checkpoint": PHOBERT_CKPT_PATH,
    "ViSoBERT checkpoint": VISOBERT_CKPT_PATH,
    "VnCoreNLP folder": VNCORE_DIR,
}

for name, path in required_paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"❌ Không tìm thấy {name}: {path}")
    print(f"✅ Tìm thấy {name}: {path}")

print("\n🎯 Tác vụ 2 hoàn tất.")

✅ Tìm thấy PhoBERT checkpoint: /content/drive/MyDrive/DACS/phobert_absa_final_loss_balanced/final_artifacts/phobert_absa_multiaspect_loss_balanced.pt
✅ Tìm thấy ViSoBERT checkpoint: /content/drive/MyDrive/DACS/visobert_absa_final_loss_balanced/final_artifacts/visobert_absa_multiaspect_loss_balanced.pt
✅ Tìm thấy VnCoreNLP folder: /content/drive/MyDrive/DACS/My_NLP_Models/vncorenlp

🎯 Tác vụ 2 hoàn tất.


In [3]:
# ============================================================
# TÁC VỤ 3: KHAI BÁO SCHEMA + ĐỊNH NGHĨA MODEL CLASSES
# ============================================================

ASPECTS = [
    "HOTEL",
    "LOCATION",
    "ROOMS",
    "FACILITIES",
    "FOOD&DRINKS",
    "SERVICE"
]

ID2LABEL = {
    0: "NONE",
    1: "POSITIVE",
    2: "NEGATIVE",
    3: "NEUTRAL",
    4: "CONFLICT"
}

NUM_ASPECTS = len(ASPECTS)
NUM_CLASSES = len(ID2LABEL)


class PhoBERTMultiAspectClassifier(nn.Module):
    def __init__(
        self,
        pretrained_model_name: str,
        num_aspects: int,
        num_classes: int,
        class_weights: torch.Tensor,
        dropout_prob: float = 0.2
    ):
        super().__init__()

        self.num_aspects = num_aspects
        self.num_classes = num_classes

        self.encoder = AutoModel.from_pretrained(pretrained_model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout_prob)
        self.classifier = nn.Linear(
            hidden_size,
            num_aspects * num_classes
        )

        self.register_buffer("class_weights", class_weights)

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None
    ):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)

        logits = self.classifier(cls_output)
        logits = logits.view(
            -1,
            self.num_aspects,
            self.num_classes
        )

        loss = None

        if labels is not None:
            labels = labels.long()
            aspect_losses = []

            for aspect_idx in range(self.num_aspects):
                loss_fn = nn.CrossEntropyLoss(
                    weight=self.class_weights[aspect_idx]
                )

                aspect_loss = loss_fn(
                    logits[:, aspect_idx, :],
                    labels[:, aspect_idx]
                )

                aspect_losses.append(aspect_loss)

            loss = torch.stack(aspect_losses).mean()

        return {
            "loss": loss,
            "logits": logits
        }


class ViSoBERTMultiAspectClassifier(nn.Module):
    def __init__(
        self,
        pretrained_model_name: str,
        num_aspects: int,
        num_classes: int,
        class_weights: torch.Tensor,
        dropout_prob: float = 0.3
    ):
        super().__init__()

        self.num_aspects = num_aspects
        self.num_classes = num_classes

        self.encoder = AutoModel.from_pretrained(pretrained_model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout_prob)
        self.classifier = nn.Linear(
            hidden_size,
            num_aspects * num_classes
        )

        self.register_buffer("class_weights", class_weights)

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None
    ):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)

        logits = self.classifier(cls_output)
        logits = logits.view(
            -1,
            self.num_aspects,
            self.num_classes
        )

        loss = None

        if labels is not None:
            labels = labels.long()
            aspect_losses = []

            for aspect_idx in range(self.num_aspects):
                loss_fn = nn.CrossEntropyLoss(
                    weight=self.class_weights[aspect_idx]
                )

                aspect_loss = loss_fn(
                    logits[:, aspect_idx, :],
                    labels[:, aspect_idx]
                )

                aspect_losses.append(aspect_loss)

            loss = torch.stack(aspect_losses).mean()

        return {
            "loss": loss,
            "logits": logits
        }


print("✅ Tác vụ 3 hoàn tất.")
print("ASPECTS:", ASPECTS)
print("ID2LABEL:", ID2LABEL)

✅ Tác vụ 3 hoàn tất.
ASPECTS: ['HOTEL', 'LOCATION', 'ROOMS', 'FACILITIES', 'FOOD&DRINKS', 'SERVICE']
ID2LABEL: {0: 'NONE', 1: 'POSITIVE', 2: 'NEGATIVE', 3: 'NEUTRAL', 4: 'CONFLICT'}


In [4]:
# ============================================================
# TÁC VỤ 4: LOAD CHECKPOINT + TOKENIZER + MODEL
# ============================================================

def safe_torch_load(path: str):
    """
    Tương thích nhiều version PyTorch.
    """
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=DEVICE)


def normalize_id2label(raw_id2label: dict) -> dict:
    """
    Đảm bảo key của ID2LABEL luôn là int.
    """
    return {int(k): v for k, v in raw_id2label.items()}


def get_dropout_from_checkpoint(ckpt: dict, default_value: float) -> float:
    best_params = ckpt.get("best_hyperparameters", {})
    try:
        return float(best_params.get("dropout_prob", default_value))
    except Exception:
        return default_value


def load_phobert_bundle(checkpoint_path: str) -> dict:
    ckpt = safe_torch_load(checkpoint_path)

    model_name = ckpt["phobert_model_name"]
    num_aspects = int(ckpt["num_aspects"])
    num_classes = int(ckpt["num_classes"])
    aspects = list(ckpt["aspects"])
    id2label = normalize_id2label(ckpt["id2label"])
    max_length = int(ckpt["max_length"])
    class_weights = torch.tensor(
        ckpt["class_weights"],
        dtype=torch.float32
    )
    dropout_prob = get_dropout_from_checkpoint(
        ckpt,
        default_value=0.2
    )

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=False
    )

    model = PhoBERTMultiAspectClassifier(
        pretrained_model_name=model_name,
        num_aspects=num_aspects,
        num_classes=num_classes,
        class_weights=class_weights,
        dropout_prob=dropout_prob
    )

    model.load_state_dict(
        ckpt["model_state_dict"],
        strict=True
    )

    model.to(DEVICE)
    model.eval()

    return {
        "display_name": "PhoBERT",
        "model": model,
        "tokenizer": tokenizer,
        "aspects": aspects,
        "id2label": id2label,
        "max_length": max_length,
        "dropout_prob": dropout_prob,
    }


def load_visobert_bundle(checkpoint_path: str) -> dict:
    ckpt = safe_torch_load(checkpoint_path)

    model_name = ckpt["model_name"]
    num_aspects = int(ckpt["num_aspects"])
    num_classes = int(ckpt["num_classes"])
    aspects = list(ckpt["aspects"])
    id2label = normalize_id2label(ckpt["id2label"])
    max_length = int(ckpt["max_length"])
    class_weights = torch.tensor(
        ckpt["class_weights"],
        dtype=torch.float32
    )
    dropout_prob = get_dropout_from_checkpoint(
        ckpt,
        default_value=0.3
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = ViSoBERTMultiAspectClassifier(
        pretrained_model_name=model_name,
        num_aspects=num_aspects,
        num_classes=num_classes,
        class_weights=class_weights,
        dropout_prob=dropout_prob
    )

    model.load_state_dict(
        ckpt["model_state_dict"],
        strict=True
    )

    model.to(DEVICE)
    model.eval()

    return {
        "display_name": "ViSoBERT",
        "model": model,
        "tokenizer": tokenizer,
        "aspects": aspects,
        "id2label": id2label,
        "max_length": max_length,
        "dropout_prob": dropout_prob,
    }


phobert_bundle = load_phobert_bundle(PHOBERT_CKPT_PATH)
visobert_bundle = load_visobert_bundle(VISOBERT_CKPT_PATH)

print("✅ Load model hoàn tất")
print(
    f"PhoBERT | max_length={phobert_bundle['max_length']} | "
    f"dropout={phobert_bundle['dropout_prob']}"
)
print(
    f"ViSoBERT | max_length={visobert_bundle['max_length']} | "
    f"dropout={visobert_bundle['dropout_prob']}"
)

config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/390M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/390M [00:00<?, ?B/s]

✅ Load model hoàn tất
PhoBERT | max_length=64 | dropout=0.2
ViSoBERT | max_length=128 | dropout=0.3


In [5]:
# ============================================================
# TÁC VỤ 5: LOAD VNCORE + CLEAN CƠ BẢN + TÁCH UNIT
# ============================================================

vncore_segmenter = py_vncorenlp.VnCoreNLP(
    annotators=["wseg"],
    save_dir=VNCORE_DIR
)


def basic_clean_text(text: str) -> str:
    """
    Clean tối giản cho input demo:
    - ép kiểu str
    - lowercase
    - chuẩn hóa xuống dòng
    - co khoảng trắng
    """
    if text is None:
        return ""

    text = str(text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.lower().strip()

    cleaned_lines = []

    for line in text.split("\n"):
        line = re.sub(r"\s+", " ", line).strip()
        if line:
            cleaned_lines.append(line)

    return "\n".join(cleaned_lines)


def split_review_to_units(text: str) -> list:
    """
    Tách câu/unit theo:
    - .
    - ?
    - !
    - xuống dòng
    """
    cleaned = basic_clean_text(text)

    if not cleaned:
        return []

    raw_units = re.split(r"[.!?]+|\n+", cleaned)

    units = []

    for unit in raw_units:
        unit = re.sub(r"\s+", " ", unit).strip(" ,;:-")
        if unit:
            units.append(unit)

    return units


def vncore_word_segment(text: str) -> str:
    """
    Chạy VnCoreNLP word segmentation cho 1 unit.
    """
    if not isinstance(text, str) or not text.strip():
        return ""

    try:
        output = vncore_segmenter.word_segment(text.strip())

        if isinstance(output, list):
            return " ".join(
                sent.strip()
                for sent in output
                if isinstance(sent, str) and sent.strip()
            )

        if isinstance(output, str):
            return output.strip()

        return text.strip()

    except Exception as e:
        print(f"⚠️ Lỗi VnCore với text: {text[:80]} | {e}")
        return text.strip()


def preprocess_review(review_text: str) -> list:
    """
    Review -> danh sách unit đã có model_input.
    """
    units = split_review_to_units(review_text)

    processed = []

    for unit_id, unit_text in enumerate(units, start=1):
        model_input = vncore_word_segment(unit_text)

        processed.append({
            "unit_id": unit_id,
            "unit_text": unit_text,
            "model_input": model_input,
        })

    return processed


# Test nhanh phần tách unit + segmentation
sample_preprocess = preprocess_review(
    "Phòng sạch sẽ. Nhân viên rất nhiệt tình! Vị trí hơi xa trung tâm?\nBữa sáng ngon"
)

display(pd.DataFrame(sample_preprocess))

print("✅ Tác vụ 5 hoàn tất.")

,unit_id,unit_text,model_input
0,1,phòng sạch sẽ,phòng sạch_sẽ
1,2,nhân viên rất nhiệt tình,nhân_viên rất nhiệt_tình
2,3,vị trí hơi xa trung tâm,vị_trí hơi xa trung_tâm
3,4,bữa sáng ngon,bữa sáng ngon


✅ Tác vụ 5 hoàn tất.


In [6]:
# ============================================================
# TÁC VỤ 6: HÀM TOKENIZE, PREDICT, DECODE CHO TỪNG MODEL
# ============================================================

def encode_units_for_bundle(model_inputs: list, bundle: dict) -> dict:
    encoded = bundle["tokenizer"](
        model_inputs,
        padding=True,
        truncation=True,
        max_length=bundle["max_length"],
        return_tensors="pt"
    )

    return {
        "input_ids": encoded["input_ids"].to(DEVICE),
        "attention_mask": encoded["attention_mask"].to(DEVICE),
    }


def decode_logits_to_annotations(
    logits: torch.Tensor,
    bundle: dict
) -> list:
    """
    logits shape: [batch, num_aspects, num_classes]
    """
    probs = torch.softmax(logits, dim=-1)
    pred_ids = torch.argmax(probs, dim=-1)

    batch_annotations = []

    for row_idx in range(pred_ids.shape[0]):
        annotations = []

        for aspect_idx, aspect in enumerate(bundle["aspects"]):
            label_id = int(pred_ids[row_idx, aspect_idx].item())
            label_name = bundle["id2label"][label_id]
            confidence = float(
                probs[row_idx, aspect_idx, label_id].item()
            )

            if label_name != "NONE":
                annotations.append({
                    "aspect": aspect,
                    "sentiment": label_name,
                    "confidence": round(confidence, 4),
                })

        batch_annotations.append(annotations)

    return batch_annotations


@torch.no_grad()
def predict_processed_units_with_model(
    processed_units: list,
    bundle: dict
) -> list:
    if not processed_units:
        return []

    model_inputs = [
        item["model_input"]
        for item in processed_units
    ]

    encoded = encode_units_for_bundle(
        model_inputs,
        bundle
    )

    outputs = bundle["model"](
        input_ids=encoded["input_ids"],
        attention_mask=encoded["attention_mask"]
    )

    logits = outputs["logits"]

    return decode_logits_to_annotations(
        logits,
        bundle
    )


print("✅ Tác vụ 6 hoàn tất.")

✅ Tác vụ 6 hoàn tất.


In [7]:
# ============================================================
# TÁC VỤ 7 SỬA LẠI: BỎ CỘT model_input
# ============================================================

def predict_review(
    review_text: str,
    model_mode: str = "both"
) -> list:
    model_mode = str(model_mode).strip().lower()

    if model_mode not in {"phobert", "visobert", "both"}:
        raise ValueError(
            "model_mode chỉ nhận: 'phobert', 'visobert', hoặc 'both'"
        )

    processed_units = preprocess_review(review_text)

    phobert_predictions = None
    visobert_predictions = None

    if model_mode in {"phobert", "both"}:
        phobert_predictions = predict_processed_units_with_model(
            processed_units,
            phobert_bundle
        )

    if model_mode in {"visobert", "both"}:
        visobert_predictions = predict_processed_units_with_model(
            processed_units,
            visobert_bundle
        )

    results = []

    for idx, unit in enumerate(processed_units):
        row = {
            "unit_id": unit["unit_id"],
            "unit_text": unit["unit_text"],
        }

        if phobert_predictions is not None:
            row["PhoBERT"] = phobert_predictions[idx]

        if visobert_predictions is not None:
            row["ViSoBERT"] = visobert_predictions[idx]

        results.append(row)

    return results


def results_to_dataframe(results: list) -> pd.DataFrame:
    rows = []

    for unit in results:
        base = {
            "unit_id": unit["unit_id"],
            "unit_text": unit["unit_text"],
        }

        model_names = [
            name for name in ["PhoBERT", "ViSoBERT"]
            if name in unit
        ]

        for model_name in model_names:
            annotations = unit.get(model_name, [])

            if not annotations:
                rows.append({
                    **base,
                    "model": model_name,
                    "aspect": "NONE",
                    "sentiment": "NONE",
                    "confidence": None,
                })
            else:
                for ann in annotations:
                    rows.append({
                        **base,
                        "model": model_name,
                        "aspect": ann["aspect"],
                        "sentiment": ann["sentiment"],
                        "confidence": ann["confidence"],
                    })

    return pd.DataFrame(rows)

print("✅ Đã cập nhật Tác vụ 7: bỏ cột model_input")

✅ Đã cập nhật Tác vụ 7: bỏ cột model_input


In [8]:
# ============================================================
# TÁC VỤ 8: TEST PIPELINE HOÀN CHỈNH
# ============================================================

demo_review = """
Phòng sạch sẽ và khá rộng.
Nhân viên thân thiện!
Vị trí hơi xa trung tâm?
Bữa sáng ngon và đa dạng.
"""

results = predict_review(
    demo_review,
    model_mode="both"
)

print("===== RAW RESULTS =====")
for item in results:
    print(item)

print("\n===== DATAFRAME VIEW =====")
display(results_to_dataframe(results))

===== RAW RESULTS =====
{'unit_id': 1, 'unit_text': 'phòng sạch sẽ và khá rộng', 'PhoBERT': [{'aspect': 'ROOMS', 'sentiment': 'POSITIVE', 'confidence': 0.9627}], 'ViSoBERT': [{'aspect': 'ROOMS', 'sentiment': 'POSITIVE', 'confidence': 0.9816}]}
{'unit_id': 2, 'unit_text': 'nhân viên thân thiện', 'PhoBERT': [{'aspect': 'SERVICE', 'sentiment': 'POSITIVE', 'confidence': 0.9764}], 'ViSoBERT': [{'aspect': 'SERVICE', 'sentiment': 'CONFLICT', 'confidence': 0.4078}]}
{'unit_id': 3, 'unit_text': 'vị trí hơi xa trung tâm', 'PhoBERT': [{'aspect': 'LOCATION', 'sentiment': 'NEGATIVE', 'confidence': 0.6764}], 'ViSoBERT': [{'aspect': 'LOCATION', 'sentiment': 'POSITIVE', 'confidence': 0.9598}]}
{'unit_id': 4, 'unit_text': 'bữa sáng ngon và đa dạng', 'PhoBERT': [{'aspect': 'FOOD&DRINKS', 'sentiment': 'POSITIVE', 'confidence': 0.9575}], 'ViSoBERT': [{'aspect': 'FOOD&DRINKS', 'sentiment': 'POSITIVE', 'confidence': 0.9925}]}

===== DATAFRAME VIEW =====


,unit_id,unit_text,model,aspect,sentiment,confidence
0,1,phòng sạch sẽ và khá rộng,PhoBERT,ROOMS,POSITIVE,0.9627
1,1,phòng sạch sẽ và khá rộng,ViSoBERT,ROOMS,POSITIVE,0.9816
2,2,nhân viên thân thiện,PhoBERT,SERVICE,POSITIVE,0.9764
3,2,nhân viên thân thiện,ViSoBERT,SERVICE,CONFLICT,0.4078
4,3,vị trí hơi xa trung tâm,PhoBERT,LOCATION,NEGATIVE,0.6764
5,3,vị trí hơi xa trung tâm,ViSoBERT,LOCATION,POSITIVE,0.9598
6,4,bữa sáng ngon và đa dạng,PhoBERT,FOOD&DRINKS,POSITIVE,0.9575
7,4,bữa sáng ngon và đa dạng,ViSoBERT,FOOD&DRINKS,POSITIVE,0.9925


# DEMO BẰNG WEB

In [9]:
# ============================================================
# TÁC VỤ 9 SỬA LẠI: OUTPUT CHO WEB DEMO
# ============================================================

!pip install -q gradio

import gradio as gr
import html


def build_prediction_summary_html(results: list, model_mode: str) -> str:
    if not results:
        return """
        <div class="empty-summary">
            Chưa có câu/unit hợp lệ để phân tích.
        </div>
        """

    total_units = len(results)

    if model_mode == "both":
        model_text = "PhoBERT + ViSoBERT"
    elif model_mode == "phobert":
        model_text = "PhoBERT"
    else:
        model_text = "ViSoBERT"

    total_annotations = 0
    detected_aspects = set()

    for item in results:
        for model_name in ["PhoBERT", "ViSoBERT"]:
            if model_name in item:
                annotations = item[model_name]
                total_annotations += len(annotations)

                for ann in annotations:
                    detected_aspects.add(ann["aspect"])

    aspect_text = ", ".join(sorted(detected_aspects)) if detected_aspects else "Không phát hiện"

    return f"""
    <div class="summary-grid">
        <div class="summary-card">
            <div class="summary-label">Số unit/câu</div>
            <div class="summary-value">{total_units}</div>
        </div>

        <div class="summary-card">
            <div class="summary-label">Mô hình</div>
            <div class="summary-value summary-model">{html.escape(model_text)}</div>
        </div>

        <div class="summary-card">
            <div class="summary-label">Aspect–Sentiment</div>
            <div class="summary-value">{total_annotations}</div>
        </div>
    </div>

    <div class="aspect-strip">
        <span class="aspect-strip-title">Aspect được phát hiện:</span>
        <span class="aspect-strip-content">{html.escape(aspect_text)}</span>
    </div>
    """


def web_predict(review_text: str, model_mode: str):
    if review_text is None or not str(review_text).strip():
        empty_df = pd.DataFrame(
            columns=[
                "unit_id",
                "unit_text",
                "model",
                "aspect",
                "sentiment",
                "confidence"
            ]
        )

        empty_html = """
        <div class="empty-summary">
            Vui lòng nhập một đoạn review trước khi phân tích.
        </div>
        """

        return empty_html, empty_df

    results = predict_review(
        review_text=review_text,
        model_mode=model_mode
    )

    result_df = results_to_dataframe(results)
    summary_html = build_prediction_summary_html(results, model_mode)

    return summary_html, result_df


print("✅ Đã cập nhật Tác vụ 9")

✅ Đã cập nhật Tác vụ 9


In [12]:
# ============================================================
# TÁC VỤ 10 SỬA LẠI: GIAO DIỆN SÁNG HƠN, KHỐI NỔI RÕ HƠN
# ============================================================

CUSTOM_CSS = """
/* =========================
   TỔNG THỂ
========================= */
body {
    background: #eef3f9 !important;
}

.gradio-container {
    max-width: 1280px !important;
    margin: 0 auto !important;
    background: #eef3f9 !important;
    color: #172033 !important;
    font-family: Inter, ui-sans-serif, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif !important;
}

/* chữ chung rõ hơn */
.gradio-container,
.gradio-container p,
.gradio-container span,
.gradio-container label,
.gradio-container h1,
.gradio-container h2,
.gradio-container h3,
.gradio-container h4 {
    color: #172033 !important;
}

/* =========================
   HERO
========================= */
.hero-card {
    background: linear-gradient(135deg, #ffffff 0%, #edf4ff 100%);
    border: 1px solid #dce6f5;
    border-radius: 24px;
    padding: 28px 30px;
    margin-bottom: 22px;
    box-shadow: 0 16px 40px rgba(26, 43, 74, 0.08);
}

.hero-eyebrow {
    display: inline-flex;
    align-items: center;
    gap: 8px;
    padding: 7px 12px;
    border-radius: 999px;
    background: #e7f0ff;
    color: #2456b4 !important;
    font-weight: 800;
    font-size: 13px;
    margin-bottom: 14px;
}

.hero-title {
    font-size: 34px;
    line-height: 1.18;
    font-weight: 850;
    color: #12203a !important;
    margin: 0 0 12px 0;
}

.hero-subtitle {
    font-size: 16px;
    line-height: 1.65;
    color: #4a5a73 !important;
    max-width: 920px;
    margin: 0 0 18px 0;
}

.hero-tags {
    display: flex;
    flex-wrap: wrap;
    gap: 10px;
}

.hero-tag {
    background: #ffffff;
    border: 1px solid #dbe5f5;
    color: #334155 !important;
    padding: 8px 12px;
    border-radius: 999px;
    font-size: 13px;
    font-weight: 700;
}

/* =========================
   CARD CHUNG
========================= */
.panel-card {
    background: #ffffff !important;
    border: 1px solid #dce4ef !important;
    border-radius: 22px !important;
    padding: 20px !important;
    box-shadow: 0 14px 34px rgba(25, 46, 89, 0.07) !important;
    margin-bottom: 18px !important;
}

.section-title {
    font-size: 20px;
    font-weight: 850;
    color: #172b4d !important;
    margin: 0 0 8px 0;
}

.section-desc {
    font-size: 14px;
    color: #5b6b83 !important;
    margin: 0 0 16px 0;
    line-height: 1.6;
}

/* đảm bảo bên trong group không bị nền tối */
.panel-card .gr-block,
.panel-card .gr-box,
.panel-card .gr-form,
.panel-card .gr-group,
.panel-card .gr-panel {
    background: transparent !important;
    border: none !important;
    box-shadow: none !important;
}

/* =========================
   INPUT
========================= */
.review-box textarea {
    min-height: 190px !important;
    border-radius: 18px !important;
    border: 1px solid #d6dfec !important;
    background: #fbfdff !important;
    color: #172033 !important;
    font-size: 15px !important;
    line-height: 1.65 !important;
}

.review-box label,
.model-dropdown label,
.result-table label {
    color: #1f2a3d !important;
    font-weight: 800 !important;
}

/* =========================
   CONTROL
========================= */
.control-row {
    margin-top: 10px;
}

.model-dropdown {
    min-height: 74px;
}

.analyze-btn {
    min-height: 74px !important;
    border-radius: 18px !important;
    font-size: 16px !important;
    font-weight: 800 !important;
    background: linear-gradient(135deg, #2563eb 0%, #1d4ed8 100%) !important;
    border: none !important;
    color: white !important;
    box-shadow: 0 14px 28px rgba(37, 99, 235, 0.24) !important;
}

.analyze-btn:hover {
    transform: translateY(-1px);
    box-shadow: 0 18px 34px rgba(37, 99, 235, 0.30) !important;
}

/* =========================
   SUMMARY
========================= */
.summary-grid {
    display: grid;
    grid-template-columns: repeat(3, minmax(0, 1fr));
    gap: 14px;
    margin-bottom: 16px;
}

.summary-card {
    background: linear-gradient(180deg, #ffffff 0%, #f8fbff 100%);
    border: 1px solid #e1e8f2;
    border-radius: 18px;
    padding: 16px 18px;
    box-shadow: 0 10px 24px rgba(25, 46, 89, 0.05);
}

.summary-label {
    color: #64748b !important;
    font-size: 13px;
    font-weight: 800;
    margin-bottom: 8px;
}

.summary-value {
    color: #14213d !important;
    font-size: 26px;
    line-height: 1.2;
    font-weight: 850;
}

.summary-model {
    font-size: 20px;
}

.aspect-strip {
    background: #eef4ff !important;
    border: 1px solid #d8e5ff !important;
    border-radius: 16px;
    padding: 14px 16px;
    color: #1d3e8a !important;
    font-size: 14px;
    line-height: 1.55;
    box-shadow: 0 6px 16px rgba(30, 64, 175, 0.05);
}

.aspect-strip-title {
    font-weight: 850;
    margin-right: 8px;
    color: #163f96 !important;
}

.aspect-strip-content {
    font-weight: 700;
    color: #1e3a8a !important;
}

.empty-summary {
    background: #fff7ed !important;
    border: 1px solid #fed7aa !important;
    color: #9a3412 !important;
    border-radius: 16px;
    padding: 16px 18px;
    font-size: 15px;
    font-weight: 700;
}

/* =========================
   DATAFRAME / TABLE
========================= */
.result-table,
.result-table > div,
.result-table .wrap,
.result-table .table-wrap {
    background: #ffffff !important;
    border-radius: 18px !important;
}

.result-table table {
    background: #ffffff !important;
    color: #1f2937 !important;
    font-size: 14px !important;
}

.result-table th {
    background: #eaf2ff !important;
    color: #16325c !important;
    font-weight: 850 !important;
}

.result-table td {
    background: #ffffff !important;
    color: #243447 !important;
}

.result-table tr:nth-child(even) td {
    background: #f8fbff !important;
}

/* =========================
   EXAMPLES
========================= */
.examples-panel,
.examples-panel > div,
.examples-panel .wrap,
.examples-panel .table-wrap {
    background: #ffffff !important;
}

.examples-panel table {
    background: #ffffff !important;
    color: #1f2937 !important;
    border-radius: 16px !important;
    overflow: hidden !important;
}

.examples-panel th {
    background: #eef4ff !important;
    color: #17325c !important;
    font-weight: 850 !important;
}

.examples-panel td {
    background: #ffffff !important;
    color: #243447 !important;
}

.examples-panel tr:nth-child(even) td {
    background: #f8fbff !important;
}

/* =========================
   RESPONSIVE
========================= */
@media (max-width: 900px) {
    .hero-title {
        font-size: 28px;
    }

    .summary-grid {
        grid-template-columns: 1fr;
    }
}
/* ============================================================
   FIX NỀN TỐI BỊ CHÌM Ở PANEL 1, 2, 3
============================================================ */

#input_panel,
#summary_panel,
#result_panel {
    background: #ffffff !important;
    border: 1px solid #dce4ef !important;
    border-radius: 22px !important;
    box-shadow: 0 14px 34px rgba(25, 46, 89, 0.07) !important;
}

#input_panel > div,
#summary_panel > div,
#result_panel > div,
#input_panel .contain,
#summary_panel .contain,
#result_panel .contain,
#input_panel .wrap,
#summary_panel .wrap,
#result_panel .wrap,
#input_panel .block,
#summary_panel .block,
#result_panel .block,
#input_panel .gr-box,
#summary_panel .gr-box,
#result_panel .gr-box,
#input_panel .gr-form,
#summary_panel .gr-form,
#result_panel .gr-form,
#input_panel .form,
#summary_panel .form,
#result_panel .form {
    background: #ffffff !important;
    color: #172033 !important;
    border-color: transparent !important;
}

#input_panel .section-title,
#summary_panel .section-title,
#result_panel .section-title {
    color: #172b4d !important;
}

#input_panel .section-desc,
#summary_panel .section-desc,
#result_panel .section-desc {
    color: #5b6b83 !important;
}

#input_panel label,
#result_panel label {
    color: #1f2a3d !important;
    font-weight: 800 !important;
}
/* ============================================================
   FIX TRIỆT ĐỂ NỀN ĐEN TRONG DATAFRAME (result-table)
============================================================ */

#result_panel .result-table,
#result_panel .result-table * {
    background-color: #ffffff !important;
    color: #243447 !important;
}

/* Gradio Dataframe dùng bảng HTML thật bên trong .table-wrap */
.result-table table,
.result-table table tbody,
.result-table table tr,
.result-table table td,
.result-table table th {
    background-color: #ffffff !important;
    background: #ffffff !important;
    color: #243447 !important;
    border-color: #e5eaf2 !important;
}

.result-table table th {
    background-color: #eaf2ff !important;
    background: #eaf2ff !important;
    color: #16325c !important;
    font-weight: 850 !important;
}

/* Zebra stripe cho cả dòng lẻ lẫn chẵn, ghi đè theme tối */
.result-table table tr:nth-child(odd) td {
    background-color: #ffffff !important;
    background: #ffffff !important;
}

.result-table table tr:nth-child(even) td {
    background-color: #f8fbff !important;
    background: #f8fbff !important;
}

/* Một số bản Gradio bọc thêm div.cell-wrap / span bên trong td */
.result-table table td > *,
.result-table table th > * {
    background: transparent !important;
    color: inherit !important;
}

/* Override biến CSS nội bộ mà Gradio Dataframe dùng cho dark mode */
#result_panel,
.result-table {
    --table-even-background-fill: #f8fbff !important;
    --table-odd-background-fill: #ffffff !important;
    --table-row-focus: #eaf2ff !important;
    --body-text-color: #243447 !important;
    --background-fill-primary: #ffffff !important;
    --background-fill-secondary: #ffffff !important;
    --border-color-primary: #e5eaf2 !important;
}
"""

try:
    demo.close()
except:
    pass

with gr.Blocks(
    title="ABSA Hotel Review Demo",
    theme=gr.themes.Default(),
    css=CUSTOM_CSS
) as demo:
    gr.HTML(
        """
        <div class="hero-card">
            <div class="hero-eyebrow">NLP Demo · Aspect-Based Sentiment Analysis</div>

            <h1 class="hero-title">
                Phân tích cảm xúc theo khía cạnh<br>
                cho review khách sạn tiếng Việt
            </h1>

            <p class="hero-subtitle">
                Nhập một đoạn review bất kỳ. Hệ thống sẽ tự tách thành từng câu/unit,
                chuẩn hóa bằng VnCoreNLP, sau đó dự đoán aspect và sentiment bằng
                PhoBERT, ViSoBERT hoặc đồng thời cả hai mô hình.
            </p>

            <div class="hero-tags">
                <span class="hero-tag">6 nhóm aspect</span>
                <span class="hero-tag">5 mức sentiment</span>
                <span class="hero-tag">PhoBERT</span>
                <span class="hero-tag">ViSoBERT</span>
                <span class="hero-tag">Unit-level output</span>
            </div>
        </div>
        """
    )

    with gr.Group(
    elem_classes=["panel-card"],
    elem_id="input_panel"
):
        gr.HTML(
            """
            <div class="section-title">1. Nhập review cần phân tích</div>
            <div class="section-desc">
                Có thể nhập một hoặc nhiều câu. Hệ thống sẽ tách theo dấu chấm, hỏi, chấm than và xuống dòng.
            </div>
            """
        )

        review_input = gr.Textbox(
            label="Nội dung review",
            placeholder=(
                "Ví dụ: Phòng sạch sẽ và khá rộng. "
                "Nhân viên thân thiện! Vị trí hơi xa trung tâm."
            ),
            lines=7,
            elem_classes=["review-box"]
        )

        with gr.Row(elem_classes=["control-row"]):
            model_selector = gr.Dropdown(
                choices=[
                    ("Cả hai mô hình", "both"),
                    ("PhoBERT", "phobert"),
                    ("ViSoBERT", "visobert"),
                ],
                value="both",
                label="Chọn mô hình phân tích",
                elem_classes=["model-dropdown"],
                scale=2
            )

            analyze_button = gr.Button(
                "Phân tích review",
                variant="primary",
                elem_classes=["analyze-btn"],
                scale=1
            )

    with gr.Group(
    elem_classes=["panel-card"],
    elem_id="summary_panel"
):
        gr.HTML(
            """
            <div class="section-title">2. Tổng quan kết quả</div>
            <div class="section-desc">
                Tóm tắt nhanh số câu được xử lý và lượng dự đoán aspect–sentiment.
            </div>
            """
        )

        summary_output = gr.HTML()

    with gr.Group(
    elem_classes=["panel-card"],
    elem_id="result_panel"
):
        gr.HTML(
            """
            <div class="section-title">3. Kết quả chi tiết theo từng unit</div>
            <div class="section-desc">
                Mỗi dòng thể hiện kết quả dự đoán của một mô hình cho một aspect trong từng câu.
            </div>
            """
        )

        dataframe_output = gr.Dataframe(
            label="Bảng dự đoán",
            interactive=False,
            max_height=420,
            elem_classes=["result-table"]
        )

    with gr.Group(elem_classes=["panel-card", "examples-panel"]):
        gr.HTML(
            """
            <div class="section-title">4. Ví dụ để thử nhanh</div>
            <div class="section-desc">
                Bấm vào một dòng mẫu để nạp nội dung vào form.
            </div>
            """
        )

        gr.Examples(
            examples=[
                [
                    "Phòng sạch sẽ và khá rộng. Nhân viên thân thiện! Vị trí hơi xa trung tâm?",
                    "both"
                ],
                [
                    "Bữa sáng ngon và đa dạng. Hồ bơi hơi bẩn.",
                    "both"
                ],
                [
                    "Khách sạn đẹp. Lễ tân phản hồi khá chậm.",
                    "phobert"
                ],
                [
                    "Vị trí gần trung tâm, đi lại thuận tiện. Phòng hơi nhỏ.",
                    "visobert"
                ],
            ],
            inputs=[review_input, model_selector]
        )

    analyze_button.click(
        fn=web_predict,
        inputs=[review_input, model_selector],
        outputs=[summary_output, dataframe_output]
    )

print("✅ Đã cập nhật Tác vụ 10")

Closing server running on port: 7860


/tmp/ipykernel_1009/140641637.py:439: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


✅ Đã cập nhật Tác vụ 10


In [13]:
# ============================================================
# TÁC VỤ 11 MỚI: LAUNCH WEB DEMO
# ============================================================

demo.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c5d5aaae22548c507f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
